# Calculator using an Expression Tree

This code implements an Expression Tree, a specific type of binary tree used to represent and evaluate mathematical expressions. It takes a standard "infix" string (like 3 + 4), converts it into a hierarchical tree structure based on math rules, and then solves it. [1, 2] 
The process follows a variation of the Shunting-yard algorithm. It uses two stacks to manage operators and operands, ensuring that "order of operations" (PEMDAS/BODMAS) is strictly followed. By building the tree, the code preserves the relationship between numbers and their operators, where leaves are numbers and internal nodes are operators. [3] 
Key Features & Functionality

* Operator Precedence & Associativity: The is_greater_precedence and associativity functions ensure that operations like multiplication happen before addition, and right-associative operations (like powers/exponents) are handled correctly.
* Tree Construction: The build_tree function parses the string and links Node objects. When an operator is processed, it pops two sub-trees from the stack, attaches them as children to the operator node, and pushes the new sub-tree back.
* Recursive Evaluation: The evaluate function travels down the tree. It recursively solves the left and right branches of every operator until it hits the numbers (leaf nodes), then works its way back up to provide the final result.
* Postorder Traversal: The postorder method prints the expression in Reverse Polish Notation (RPN). This is a linear way of writing math where the operator always follows its operands (e.g., 3 4 + instead of 3 + 4). [4, 5] 

[1] [https://www.cliffsnotes.com](https://www.cliffsnotes.com/cliffs-questions/5064783#:~:text=This%20implementation%20is%20a%20practical%20demonstration%20of,constructed%20and%20traversed%20in%20a%20programming%20context.)
[2] [https://www.scribd.com](https://www.scribd.com/document/904785552/241csc301j-Workbook-Lineards#:~:text=Takes%20a%20valid%20infix%20expression%20as%20input.)
[3] [https://www2.seas.gwu.edu](https://www2.seas.gwu.edu/~simhaweb/cs1112/modules/module10/suppl/index.html)
[4] [https://leanprover.github.io](https://leanprover.github.io/functional_programming_in_lean/monads.html#:~:text=The%20exponentiation%20operator%20%5E%20is%20right%20associative%2C,a%20syntax%20error%20and%20requires%20manual%20parentheses.)
[5] [https://www.cliffsnotes.com](https://www.cliffsnotes.com/cliffs-questions/5064783#:~:text=If%20the%20character%20is%20an%20operator%2C%20it,the%20new%20node%20back%20onto%20the%20stack.)


In [1]:
import operator
class Node:
    def __init__(self, value):
        self.left = None
        self.data = value
        self.right = None

    def postorder(self):

        if self.left:
            self.left.postorder()
        if self.right:
            self.right.postorder()
        print(self.data, end=" ")

def is_greater_precedence(op1, op2):
    pre = {'+':0, '-': 0, '*':1, '/':1, '^':2}
    return pre[op1] >= pre[ op2]


def associativity(op):
    assoc = {'+':0, '-':0, '*':0, '/':0, '^':1}
    return assoc[op]


def build_tree(exp):
    exp_list = exp.split()
    print(exp_list)
    stack = []
    tree_stack = []
    for i in exp_list:
        if i not in ['+', '-', '*', '/', '^', '(', ')']:
            t = Node(int(i))
            tree_stack.append(t)

        elif i in ['+', '-', '*', '/', '^']:
                if not stack or stack[-1] == '(':
                    stack.append(i)

                elif is_greater_precedence(i, stack[-1]) and           associativity(i) == 1:
                    stack.append(i)

                else:
                    while stack and is_greater_precedence(stack[-1], i) and associativity(i) == 0:
                        popped_item = stack.pop()
                        t = Node(popped_item)
                        t1 = tree_stack.pop()
                        t2 = tree_stack.pop()
                        t.right = t1
                        t.left = t2
                        tree_stack.append(t)
                    stack.append(i)

        elif i == '(':
            stack.append('(')

        elif i == ')':
            while stack[-1] != '(':
                popped_item = stack.pop()
                t = Node(popped_item)
                t1 = tree_stack.pop()
                t2 = tree_stack.pop()
                t.right = t1
                t.left = t2
                tree_stack.append(t)
            stack.pop()
        print(f"stack = {stack}, tree_stack = {[e.data for e in tree_stack]}")
    while stack:
        print(stack)
        popped_item = stack.pop()
        t = Node(popped_item)
        t1 = tree_stack.pop()
        t2 = tree_stack.pop()
        t.right = t1
        t.left = t2
        tree_stack.append(t)

    t = tree_stack.pop()

    return t

def evaluate(expTree):
    opers = {'+':operator.add, '-':operator.sub, '*':operator.mul, '/':operator.truediv, '^':operator.pow}

    leftC = expTree.left
    rightC = expTree.right

    if leftC and rightC:
        fn = opers[expTree.data]
        return fn(evaluate(leftC), evaluate(rightC))
    else:
        return expTree.data

t = build_tree("3 + 4 * 2 / ( 1 / 5 ) ^ 2 ^ 3")
print(evaluate(t))
t.postorder()

['3', '+', '4', '*', '2', '/', '(', '1', '-', '5', ')', '^', '2', '^', '3']
stack = [], tree_stack = [3]
stack = ['+'], tree_stack = [3]
stack = ['+'], tree_stack = [3, 4]
stack = ['+', '*'], tree_stack = [3, 4]
stack = ['+', '*'], tree_stack = [3, 4, 2]
stack = ['+', '/'], tree_stack = [3, '*']
stack = ['+', '/', '('], tree_stack = [3, '*']
stack = ['+', '/', '('], tree_stack = [3, '*', 1]
stack = ['+', '/', '(', '-'], tree_stack = [3, '*', 1]
stack = ['+', '/', '(', '-'], tree_stack = [3, '*', 1, 5]
stack = ['+', '/'], tree_stack = [3, '*', '-']
stack = ['+', '/', '^'], tree_stack = [3, '*', '-']
stack = ['+', '/', '^'], tree_stack = [3, '*', '-', 2]
stack = ['+', '/', '^', '^'], tree_stack = [3, '*', '-', 2]
stack = ['+', '/', '^', '^'], tree_stack = [3, '*', '-', 2, 3]
['+', '/', '^', '^']
['+', '/', '^']
['+', '/']
['+']
3.0001220703125
3 4 2 * 1 5 - 2 3 ^ ^ / + 

In [2]:
Tree = build_tree("1 * 2 + 3 * 4")

['1', '*', '2', '+', '3', '*', '4']
stack = [], tree_stack = [1]
stack = ['*'], tree_stack = [1]
stack = ['*'], tree_stack = [1, 2]
stack = ['+'], tree_stack = ['*']
stack = ['+'], tree_stack = ['*', 3]
stack = ['+', '*'], tree_stack = ['*', 3]
stack = ['+', '*'], tree_stack = ['*', 3, 4]
['+', '*']
['+']


In [3]:
Tree.right.data

'*'

In [4]:
def evaluate(expression):
  # splitting expression at whitespaces
  expression = expression.split()
    
  # stack
  stack = []
    
  # iterating expression
  for ele in expression:
    print(ele)
    print(stack)
      
    # ele is a number
    if ele not in '/*+-':
      stack.append(int(ele))
      
    # ele is an operator
    else:
      # getting operands
      right = stack.pop()
      left = stack.pop()
        
      # performing operation according to operator
      if ele == '+':
        stack.append(left + right)
          
      elif ele == '-':
        stack.append(left - right)
          
      elif ele == '*':
        stack.append(left * right)
          
      elif ele == '/':
        stack.append(int(left / right))
    
  # return final answer.
  return stack.pop()
  
# input expression
arr = "10 6 9 3 + -11 * / * 17 + 7 +"
  
# calling evaluate()
answer = evaluate(arr)
# printing final value of the expression
print(f"Value of given expression'{arr}' = {answer}")

10
[]
6
[10]
9
[10, 6]
3
[10, 6, 9]
+
[10, 6, 9, 3]
-11
[10, 6, 12]
*
[10, 6, 12, -11]
/
[10, 6, -132]
*
[10, 0]
17
[0]
+
[0, 17]
7
[17]
+
[17, 7]
Value of given expression'10 6 9 3 + -11 * / * 17 + 7 +' = 24
